# Attention 메커니즘 직접 구현 - 실습 코드 1: 완전한 Self-Attention 구현

- Tutorial ID: `expand-attention-from-scratch`
- Tutorial: Attention 메커니즘 직접 구현
- Section ID: `expand-attention-from-scratch-code-1`
- Section: 실습 코드 1: 완전한 Self-Attention 구현


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: 완전한 Self-Attention 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
        scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
                scores = np.where(mask == 0, -1e9, scores)
        weights = softmax(scores)
        output = weights @ V
    return output, weights

class MultiHeadAttention:
        def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.W_Q = [np.random.randn(d_model, self.d_k) * 0.1 for _ in range(num_heads)]
        self.W_K = [np.random.randn(d_model, self.d_k) * 0.1 for _ in range(num_heads)]
        self.W_V = [np.random.randn(d_model, self.d_k) * 0.1 for _ in range(num_heads)]
        self.W_O = np.random.randn(d_model, d_model) * 0.1

        def forward(self, X):
        heads = []
                for h in range(self.num_heads):
            Q, K, V = X @ self.W_Q[h], X @ self.W_K[h], X @ self.W_V[h]
            head_out, _ = scaled_dot_product_attention(Q, K, V)
            heads.append(head_out)
        multi_head = np.concatenate(heads, axis=-1)
        return multi_head @ self.W_O

# 테스트
mha = MultiHeadAttention(d_model=8, num_heads=2)
X = np.random.randn(4, 8)
output = mha.forward(X)
print(f"Multi-Head Attention: {X.shape} → {output.shape}")